# Gold — Fato de Diárias

Esta tabela representa a camada analítica detalhada das diárias corporativas.

A fato especializada preserva atributos como período, quantidade de diárias e valor pago, permitindo análises específicas que não pertencem à fato consolidada de viagens.

In [0]:
from pyspark.sql import functions as F

df_diarias_silver = spark.table(
    "viagens_teste.silver.diarias"
)

print(
    "Registros Silver:",
    df_diarias_silver.count()
)

### Construção da fato de diárias

Selecionamos os atributos necessários para análises financeiras, temporais e operacionais das diárias corporativas.

In [0]:
df_fato_diarias = (
    df_diarias_silver
    .select(
        F.col("id_diaria"),
        F.col("viajante"),

        F.col("data_inicio"),
        F.col("data_fim"),

        F.col("centro_custo"),
        F.col("diretoria"),

        F.col("qtd_dias_informado"),
        F.col("qtd_dias_calculado"),

        F.col("valor_numerico").alias("valor"),

        F.col("agencia"),
        F.col("trimestre"),

        F.lit("Diaria").alias("tipo_despesa")
    )
)

In [0]:
display(df_fato_diarias)

### Validação da Gold

Verificamos a quantidade de registros e a consistência entre quantidade informada e quantidade calculada de diárias.

In [0]:
print(
    "Total de registros:",
    df_fato_diarias.count()
)

print(
    "Centro de custo ausente:",
    df_fato_diarias
    .filter(F.col("centro_custo").isNull())
    .count()
)

print(
    "Divergências entre dias informado e calculado:",
    df_fato_diarias
    .filter(
        F.col("qtd_dias_informado") !=
        F.col("qtd_dias_calculado")
    )
    .count()
)

In [0]:
(
    df_fato_diarias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.gold.fato_diarias"
    )
)

In [0]:
display(
    spark.table(
        "viagens_teste.gold.fato_diarias"
    )
)